In [ ]:
# -*- coding: utf-8 -*-
import sys
import os
import gc
import traceback
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

# ==============================================================================
# 1. PERTAHANAN ANTI-LEAK UNTUK MACBOOK M3 PRO
# ==============================================================================
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU') 

from tensorflow import keras
from tensorflow.keras import backend as K
# ==============================================================================

BASE_REP = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo'
if BASE_REP not in sys.path:
    sys.path.append(BASE_REP)

from Library import utils, dataset

# KOREKSI METODOLOGI: Zhi Geng murni menggunakan detrending tanpa bandpass filter tambahan!
# Fungsi bandpass_filter_1c dihapus untuk menjamin kemurnian arsitektur.

if __name__ == "__main__":
    CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
    HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
    SAVE_DIR = '/Volumes/Extreme SSD/stream_stead/output'
    
    if not os.path.exists(SAVE_DIR): os.makedirs(SAVE_DIR)
    
    TIMESTAMP_START = datetime.now().strftime("%Y%m%d_%H%M")
    LOG_CSV_PATH = os.path.join(SAVE_DIR, f"pure_zhi_geng_predictions_log_{TIMESTAMP_START}.csv")
    
    with open(LOG_CSV_PATH, 'w') as f:
        f.write("true_label,pred_label\n")

    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    print("[INFO] Memuat Model dan Kurva PDF 1D Murni Zhi Geng (MODE CPU)...")
    try:
        with tf.device('/CPU:0'):
            embedding_model = keras.models.load_model(filepath=MODEL_PATH)
        embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
        embeddings_Z_PDFs = utils.embedding_PDFs_1D(embedding_Z)
    except Exception as e:
        print(f"❌ Gagal memuat Model/JSON: {e}")
        sys.exit(1)

    print("[INFO] Membaca metadata STEAD...")
    df = pd.read_csv(CSV_PATH, low_memory=False)
    
    # KOREKSI FILTER DATA: Hanya memuat kategori 'earthquake_local' dan 'noise' (Abaikan QB secara total)
    df = df[df['trace_category'].isin(['earthquake_local', 'noise'])]
    
    num_points = 700 # Jendela kaku 7 detik @ 100 Hz
    BUFFER_SIZE = 256 
    
    buffer_waves = []
    buffer_labels = []

    print("\n🚀 MEMULAI REPLIKASI MURNI STREAM-TO-DISK DENGAN CPU...")
    with h5py.File(HDF5_PATH, 'r') as f_h5:
        data_group = f_h5['data']
        
        for i, (idx, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="STEAD 100K Murni Zhi Geng")):
            try:
                trace_id = row['trace_name']
                category = row['trace_category']
                wave_data = data_group[trace_id][()]
                
                # Pemotongan komponen Z (Indeks 2) murni berdasarkan protokol paper
                if category == 'earthquake_local':
                    p_arrival = int(row['p_arrival_sample'])
                    start_idx = p_arrival - 50 # 0.5 detik sebelum P-arrival
                    end_idx = start_idx + num_points
                    if end_idx > 6000 or start_idx < 0: continue
                    z_component = wave_data[start_idx:end_idx, 2] 
                    true_label = 1 # Gempa / LE
                else:
                    z_component = wave_data[:num_points, 2] 
                    true_label = 0 # Bising / NO
                
                # PRE-PROCESSING MURNI ZHI GENG: Hanya detrending & normalisasi maksimum
                z_component = z_component - np.mean(z_component) # Detrending
                norm_val = np.max(np.abs(z_component))
                if norm_val > 0: 
                    z_component /= norm_val # Normalisasi
                
                buffer_waves.append(z_component)
                buffer_labels.append(true_label)
                
                if len(buffer_waves) == BUFFER_SIZE:
                    batch_input = np.array(buffer_waves).astype(np.float32).reshape(-1, num_points, 1)
                    
                    with tf.device('/CPU:0'):
                        embeddings = embedding_model.predict_on_batch(batch_input)
                    
                    emb_T = embeddings.T 
                    like_noise = embeddings_Z_PDFs["noise"].pdf(emb_T)
                    like_le = embeddings_Z_PDFs["le"].pdf(emb_T)
                    
                    # Bandingkan murni likelihood biner antara NO dan LE
                    likelihoods = np.vstack([like_noise, like_le])
                    preds = np.argmax(likelihoods, axis=0) # 0 = Noise, 1 = LE
                    
                    with open(LOG_CSV_PATH, 'a') as f_out:
                        for t_lbl, p_lbl in zip(buffer_labels, preds):
                            f_out.write(f"{t_lbl},{p_lbl}\n")
                    
                    buffer_waves.clear()
                    buffer_labels.clear()
                    del batch_input, embeddings, emb_T, likelihoods, preds
                    
                if (i + 1) % 100000 == 0:
                    gc.collect() 
                    K.clear_session()
                    
            except Exception:
                continue

        # Eksekusi sisa buffer akhir
        if len(buffer_waves) > 0:
            batch_input = np.array(buffer_waves).astype(np.float32).reshape(-1, num_points, 1)
            with tf.device('/CPU:0'):
                embeddings = embedding_model.predict_on_batch(batch_input)
            emb_T = embeddings.T 
            like_noise = embeddings_Z_PDFs["noise"].pdf(emb_T)
            like_le = embeddings_Z_PDFs["le"].pdf(emb_T)
            
            likelihoods = np.vstack([like_noise, like_le])
            preds = np.argmax(likelihoods, axis=0)
            
            with open(LOG_CSV_PATH, 'a') as f_out:
                for t_lbl, p_lbl in zip(buffer_labels, preds):
                    f_out.write(f"{t_lbl},{p_lbl}\n")

    # ==========================================================================
    # TAHAP AKHIR: EVALUASI METRIK SELESAI
    # ==========================================================================
    print("\n[INFO] Mengumpulkan data dari SSD untuk kalkulasi metrik murni...")
    df_results = pd.read_csv(LOG_CSV_PATH)
    
    matrix, metrics = utils.calc_confusion_metrics(df_results['true_label'].tolist(), df_results['pred_label'].tolist())
    
    dataset.save_json_data(os.path.join(SAVE_DIR, f"PURE_ZHI_GENG_STEAD_1C_{TIMESTAMP_START}.json"), metrics)
    fig = utils.plot_confusion("MCU-Quake Murni (Zhi Geng 1C - NO vs LE)", ["NO", "LE"], matrix, metrics)
    fig.savefig(os.path.join(SAVE_DIR, f"Pure_ZhiGeng_Matrix_1C_{TIMESTAMP_START}.jpg"), dpi=300)
    
    print(f"\n[SUKSES] Replikasi Murni Selesai!")
    print(f"Akurasi Akhir Baseline Global: {metrics.get('Accuracy (avg.)')}")

[INFO] Memuat Model dan Kurva PDF 1D Murni Zhi Geng (MODE CPU)...
[INFO] Membaca metadata STEAD...

🚀 MEMULAI REPLIKASI MURNI STREAM-TO-DISK DENGAN CPU...


STEAD 100K Murni Zhi Geng: 100%|██████████| 1265657/1265657 [3:06:37<00:00, 113.03it/s]   
